In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 1. Basit Prompt Zinciri (Simple Prompt Chain)

* Amaç: Öğrencilere PromptTemplate ve LLMChain kavramlarını tanıtmak.
* Senaryo: Öğrenci bir metin girer ve sistem bu metni özetler.
* Kazandırılan: Zincir mantığı, input/output yapıları.
* Kullanılan Bileşenler: PromptTemplate, LLMChain

In [ ]:
# ✅ Kurulum (Kaggle)
!pip install -q "langchain>=0.2" "langchain-community>=0.2" transformers accelerate sentencepiece


In [ ]:

# ✅ Model ve Pipeline: Hugging Face (TinyLlama 1.1B Chat)
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Hafif ve açık kaynak

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    # CPU'da güvenli varsayılan; GPU varsa accelerate device_map="auto" ile yönetecek
)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=190,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ✅ Prompt Template + LLMChain
prompt_template = PromptTemplate(
    input_variables=["text"],
    template=(
        "Aşağıdaki metni 3-4 cümlede, açık ve akademik bir Türkçe ile özetle.\n\n"
        "Metin:\n{text}\n\n"
        "Özet:"
    ),
)

chain = LLMChain(llm=llm, prompt=prompt_template)

# ✅ Deneme girdisi (isterseniz input() ile kullanıcıdan da alabilirsiniz)
metin = """
LangChain, büyük dil modellerini uygulama geliştiriciler için daha işlevsel hale getiren bir çatı sunar.
Zincirler, bellek ve araç entegrasyonları sayesinde LLM'ler tekli prompt-yanıt çevriminden çıkarak çok adımlı iş akışları yürütebilir.
Örneğin RAG mimarisiyle belgelerden bilgi çekip bağlamsal yanıt üretmek mümkündür.
"""

# ✅ Çalıştır ve yazdır
output = chain.run(metin)
print("⏬ Özet:\n")
print(output)


## 2. PDF'den Bilgi Çekip Yanıtlama (Document QA)
* Amaç: Öğrencilere veri alma ve RetrievalQA yapısını tanıtmak.
* Senaryo: Öğrenci bir PDF yükler ve belge içeriğine soru sorar.
* Kazandırılan: Belge parçalama, vektörleştirme, arama.
* Kullanılan Bileşenler: PyPDFLoader, FAISS, RetrievalQA

In [ ]:
# ============================================================
# 1) Kurulum
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" "langchain-text-splitters>=0.2" \
               transformers accelerate sentencepiece \
               pypdf faiss-cpu sentence-transformers

# ============================================================
# 2) İçe Aktarımlar
# ============================================================
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import os, textwrap

# ============================================================
# 3) Model Tanımları (LLM + Embedding)
# ============================================================
# --- LLM: Hafif ve açık kaynak ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Alternatif: "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.1,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# --- Embedding: sentence-transformers (hafif ve güçlü) ---
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# ============================================================
# 4) PDF Yükleme ve Parçalama
# ============================================================
# Kaggle'da dosyayı /kaggle/input/... altına koyabilir veya manuel yükleyebilirsiniz.
PDF_PATH = "/kaggle/input/ozgecmis/CV_2025_industry_academic.pdf"  # <-- kendi PDF yolunuzu girin

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

# Parçalama: RAG için kısa-örtüşmeli segmentler önerilir.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,        # içerik yoğunluğuna göre 500-1200 arası iyi çalışır
    chunk_overlap=120,     # bağlam taşımak için küçük bir örtüşme
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(docs)

print(f"Toplam belge sayfası: {len(docs)}, üretilen parça sayısı: {len(chunks)}")

# ============================================================
# 5) Vektör Mağazası (FAISS) Oluşturma
# ============================================================
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 4})  # en ilgili 4 parçayı getir

# ============================================================
# 6) Sorgu Promptu ve RetrievalQA Zinciri
# ============================================================
# LLM’in RAG çıktısını düzenli ve kaynak-destekli üretmesi için basit bir şablon
prompt_template = PromptTemplate.from_template(
    textwrap.dedent("""
    Aşağıda bir kullanıcı sorusu ve belgeden getirilen ilgili parçalar (context) verilmektedir.
    Sadece bu bağlamdan yararlanarak, akademik ve açık bir Türkçe ile cevap ver.
    Bağlamda yoksa "Metinde bu bilgi yer almıyor." de. Varsayım yapma.

    Soru: {question}

    Bağlam:
    {context}

    Cevap:
    """)
)

# RetrievalQA, varsayılan olarak context ekler; ama çıktıyı kontrol etmek için promptu elle verebiliriz.
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",             # kısa context'ler için "stuff" yeterli
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True     # kaynak parçaları geri alalım (denetim için)
)

# ============================================================
# 7) Örnek Sorgu
# ============================================================
question = "Belgede anlatılan ana problem ve önerilen çözüm yaklaşımı nedir? Kısa özetler misin?"
result = qa_chain({"query": question})

print("\n⏬ Yanıt:\n")
print(result["result"])

print("\n📚 Kullanılan Kaynak Parçalar (Özet):\n")
for i, d in enumerate(result["source_documents"]):
    print(f"--- Parça {i+1} (s. {d.metadata.get('page', 'NA')}) ---")
    print(textwrap.shorten(d.page_content.replace("\n", " "), width=300, placeholder="..."))


## 3. Ajan ile Araç Kullanımı (Agent + Tools)
* Amaç: LangChain agent mimarisini öğretmek.
* Senaryo: Ajan, bir hesap makinesi veya arama motoru aracını nasıl ve ne zaman kullanacağına karar verir.
* Kazandırılan: Araç tanımı, dinamik karar verme.
* Kullanılan Bileşenler: initialize_agent, Tool, AgentType

In [ ]:
# ============================================================
# 1) KURULUM (Kaggle)
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-experimental>=0.0.64" \
               transformers accelerate sentencepiece duckduckgo-search


In [ ]:

# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

from langchain.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# ============================================================
# 3) LLM: Açık kaynak model
#    Not: Qwen 0.5B ReAct uyumu TinyLlama'dan genelde daha iyi.
# ============================================================
# MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # önerilen

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    top_p=0.9,
    repetition_penalty=1.05,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) ARAÇLAR: Sadece DuckDuckGo (döngü ve parse hatalarını azaltır)
# ============================================================
ddg = DuckDuckGoSearchAPIWrapper(max_results=3)   # sonuçları sınırlayalım
search_tool = DuckDuckGoSearchRun(api_wrapper=ddg, name="WebSearch")
tools = [search_tool]

# ============================================================
# 5) AGENT: ZERO_SHOT_REACT_DESCRIPTION
#    - max_iterations: az tut
#    - early_stopping_method="force": Final Answer parse edilmeden durdur
# ============================================================
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=2,
    early_stopping_method="force"
)

# ============================================================
# 6) ÇALIŞTIR
# ============================================================
print("\n===== Örnek: Web araması =====\n")
question = "OpenAI kurucuları kimlerdir?"
answer = agent.run(question)
print("\n=== Cevap ===\n", answer)


## 4. Konuşma Hafızası (Conversational Memory)
* Amaç: bir sohbetin bağlamını hatırlayan sistem oluşturması.
* Senaryo: Kullanıcı chatbot ile sohbet eder, önceki soruları hatırlayarak cevap verir.
* Kazandırılan: ConversationBufferMemory ve ChatPromptTemplate kullanımı.
* Kullanılan Bileşenler: ConversationChain, Memory

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               transformers accelerate sentencepiece

In [ ]:


# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory

# ============================================================
# 3) LLM: HF ÜZERİNDEN AÇIK KAYNAK MODEL
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # Alternatif: 

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,     # daha tutarlı cevaplar için düşük sıcaklık
    top_p=0.9,
    repetition_penalty=1.05,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) HAFIZA SEÇENEKLERİ
# ============================================================
# 4a) Tam tampon (tüm geçmişi tutar) — eğitim için en görünür seçenek
buffer_memory = ConversationBufferMemory(
    memory_key="history",      # prompt'a eklenecek değişken adı
    input_key="input",         # kullanıcının mesajı
    return_messages=False      # True seçilirse ChatMessage objeleri tutulur
)

# 4b) (Opsiyonel) Sadece son N turu tutan hafıza
# window_memory = ConversationBufferWindowMemory(k=3, memory_key="history", input_key="input")

# 4c) (Opsiyonel) Özetleyerek hafıza — uzun diyaloğu kompakt özetler
# summary_memory = ConversationSummaryMemory(llm=llm, memory_key="history", input_key="input")

# ============================================================
# 5) KONUŞMA ZİNCİRİ
# ============================================================
# Varsayılan prompt: tarihçe + input'u birleştirip LLM'e verir.
# Özel bir prompt gerekmeden, hafıza otomatik enjekte edilir.
conv = ConversationChain(
    llm=llm,
    memory=buffer_memory,   # isterseniz window_memory veya summary_memory ile değiştirin
    verbose=True            # adımları (prompt+history) görmek için
)

# ============================================================
# 6) DENEYSEL DİYALOG (hafızayı test edelim)
# ============================================================
turns = [
    "Merhaba, ben finansal zaman serileri üzerinde çalışan bir araştırmacıyım.",
    "Bugün İstanbul'da hava nasıl bilmiyorum ama toplantım var. Bana hitap ederken kısaca 'hocam' diyebilirsin.",
    "Az önce bahsettiğim çalışma alanımı tek cümlede özetler misin?",
    "Bana nasıl hitap etmen gerektiğini hatırlıyor musun?",
    "Toplantıdan sonra hangi konuyu okumamı önermiştin? (Eğer önermediysen, şimdi öner.)"
]

for i, user_msg in enumerate(turns, 1):
    print(f"\n=== Kullanıcı ({i}) ===\n{user_msg}\n")
    reply = conv.predict(input=user_msg)
    print(f"--- Asistan ---\n{reply}")

# ============================================================
# 7) HAFIZAYI GÖRÜNTÜLEME (inceleme amaçlı)
# ============================================================
print("\n\n=== Hafıza (raw buffer) ===")
print(buffer_memory.buffer)

# Eğer window_memory veya summary_memory seçerseniz:
# print(window_memory.buffer) ya da print(summary_memory.buffer)


## 5. Koşullu Akışlar (SequentialChain / ConditionalChain)
* Amaç: LLM’lerin çok aşamalı işlemleri nasıl zincirlediğini göstermek.
* Senaryo: Kullanıcıdan konu alınır → özet çıkarılır → başlık üretilir → tweet formatında yazılır.
* Kazandırılan: Çok adımlı işlem mantığı.
* Kullanılan Bileşenler: SimpleSequentialChain, LLMChain

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               transformers accelerate sentencepiece

In [ ]:

# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

import textwrap

# ============================================================
# 3) LLM: HF ÜZERİNDEN AÇIK KAYNAK MODEL
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Alternatif: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=220,
    do_sample=True,
    temperature=0.2,      # eğitimde tutarlı çıktılar için düşük sıcaklık
    top_p=0.9,
    repetition_penalty=1.05,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) PROMPTLAR ve ZİNCİRLER
# ============================================================
# 4.1 ÖZET
summary_prompt = PromptTemplate(
    input_variables=["topic"],
    template=textwrap.dedent("""\
        Aşağıdaki konuyu 3-5 cümlede, açık ve akademik Türkçe ile özetle.
        Gereksiz tekrar ve retorikten kaçın; somut kavramları vurgula.

        Konu: {topic}

        Özet:
    """)
)
summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="summary")

# 4.2 BAŞLIK
title_prompt = PromptTemplate(
    input_variables=["summary"],
    template=textwrap.dedent("""\
        Aşağıdaki özeti temel alarak, en fazla 60 karakterlik,
        teknik fakat çekici bir Türkçe başlık üret. Nokta kullanma.

        Özet:
        {summary}

        Başlık:
    """)
)
title_chain = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

# 4.3 TWEET (<= 240 karakter)
tweet_prompt = PromptTemplate(
    input_variables=["title", "summary"],
    template=textwrap.dedent("""\
        Aşağıdaki başlık ve özete dayanarak 240 karakteri geçmeyen,
        bilgilendirici bir Türkçe tweet yaz. 1-2 uygun hashtag ekle.

        Başlık: {title}
        Özet: {summary}

        Tweet:
    """)
)
tweet_chain = LLMChain(llm=llm, prompt=tweet_prompt, output_key="tweet")

# 4.4 (OPSİYONEL / KOŞULLU) TL;DR
tldr_prompt = PromptTemplate(
    input_variables=["summary"],
    template=textwrap.dedent("""\
        Aşağıdaki özeti tek cümlede TL;DR biçiminde yoğunlaştır.
        Öz cümleyi üret; 30 kelimeyi geçmesin.

        Özet:
        {summary}

        TL;DR:
    """)
)
tldr_chain = LLMChain(llm=llm, prompt=tldr_prompt, output_key="tldr")

# ============================================================
# 5) KOŞULLU AKIŞ YÜRÜTÜCÜ
# ============================================================
def run_pipeline(topic: str, tldr_threshold: int = 600):
    # Adım 1: Özet
    summary = summary_chain.run(topic).strip()

    # Adım 2: Başlık
    title = title_chain.run(summary).strip()

    # Adım 3: Tweet
    tweet = tweet_chain.run(title=title, summary=summary).strip()

    # Adım 4 (KOŞULLU): Özet çok uzunsa TL;DR üret
    # Burada metin uzunluğunu karakter bazlı kontrol ediyoruz (eğitsel amaçlı basit kural).
    if len(summary) > tldr_threshold:
        tldr = tldr_chain.run(summary=summary).strip()
    else:
        tldr = "Özet yeterince kısa; TL;DR gerekli görülmedi."

    return {
        "summary": summary,
        "title": title,
        "tweet": tweet,
        "tldr": tldr,
        "summary_len": len(summary)
    }

# ============================================================
# 6) DEMO
# ============================================================
topic = """\
Büyük dil modellerinde (LLM) akıl yürütme kabiliyetini artırmak için kullanılan
Chain-of-Thought (CoT), Self-Consistency ve Toolformer benzeri yöntemlerin
karşılaştırılması; parametre ölçeği, veri kalitesi ve yönlendirme (prompting)
stratejilerinin sonuç doğruluğuna etkisi.
"""

result = run_pipeline(topic, tldr_threshold=550)

print("\n=== ÖZET ({} karakter) ===\n".format(result["summary_len"]))
print(result["summary"])

print("\n=== BAŞLIK ===\n")
print(result["title"])

print("\n=== TWEET (<=240) ===\n")
print(result["tweet"])

print("\n=== TL;DR (Koşullu) ===\n")
print(result["tldr"])


## 6. Arama Motoru ile Sorgu (SerpAPI + Agent)
* Amaç: LLM’in internette araştırma yapma yeteneğini göstermek.
* Senaryo: Kullanıcı güncel bir soruya cevap arar, model SerpAPI aracını kullanır.
* Kazandırılan: Web tabanlı ajan etkileşimi.
* Kullanılan Bileşenler: SerpAPIWrapper, Agent, Tool

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-experimental>=0.0.64" \
               duckduckgo-search \
               transformers accelerate sentencepiece

In [ ]:


# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

from langchain.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.tools.python.tool import PythonREPLTool  # İsteğe bağlı

# ============================================================
# 3) LLM MODELİ: Açık kaynak Qwen
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # Alternatif: TinyLlama 1.1B

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) ARAMA ARACI: DuckDuckGo
# ============================================================
search_tool = DuckDuckGoSearchRun(name="WebSearch")

# (İsteğe Bağlı) Python hesaplama aracı
python_tool = PythonREPLTool(name="Python")

# ============================================================
# 5) AGENT TANIMI
# ============================================================
agent = initialize_agent(
    tools=[search_tool],  # gerekirse: + python_tool
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
)

# ============================================================
# 6) SORU & YÜRÜTME
# ============================================================
question = "Türkiye Cumhuriyet Merkez Bankası başkanı kimdir?"

print(f"\n=== Soru ===\n{question}")
response = agent.run(question)

print("\n=== Cevap ===")
print(response)


## 7. CSV Dosyası Sorgulama (CSV Agent)
* Amaç:  Veri üzerinde doğal dil sorgusu yapmayı öğretmek.
* Senaryo: Kullanıcı bir CSV dosyası yükler ve "en çok satan ürün nedir?" gibi sorular sorar.
* Kazandırılan: Veri analizi + LLM entegrasyonu.
* Kullanılan Bileşenler: create_csv_agent, PandasAgent

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               pandas transformers accelerate sentencepiece


In [ ]:

# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
import pandas as pd
from langchain_community.llms import HuggingFacePipeline
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents import create_pandas_dataframe_agent

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ============================================================
# 3) MODEL: Açık kaynak Qwen
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) CSV YÜKLEME
# ============================================================
# Kaggle'a CSV dosyanızı yükleyin, path aşağıdaki gibi olmalı:
CSV_PATH = '/kaggle/input/retail-data-set/file_out2.csv'  # burayı kendi dosyanıza göre değiştirin
df = pd.read_csv(CSV_PATH)

print("✅ Veri kümesi ilk 5 satır:")
display(df.head())

# ============================================================
# 5) CSV AGENT OLUŞTURMA
# ============================================================
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# ============================================================
# 6) DOĞAL DİL SORGULAR
# ============================================================
questions = [
    "En çok satış yapılan ürün hangisidir?",
    "Toplam gelir ne kadar?",
    "En düşük memnuniyet puanı kaç ve hangi müşteri verdi?",
    "2023 yılında kaç adet sipariş verilmiş?",
    "Fiyatı 100 TL'den fazla olan ürünlerin ortalama adedi kaç?"
]

for i, q in enumerate(questions, 1):
    print(f"\n--- Soru {i}: {q} ---")
    try:
        answer = agent.run(q)
        print(f"Cevap: {answer}")
    except Exception as e:
        print(f"Hata: {e}")


## 8. RAG Uygulaması (Retrieval-Augmented Generation)
* Amaç: LangChain ile RAG mimarisini öğretmek.
* Senaryo: Özgün bir metin koleksiyonundan bilgi çekerek modelle cevap oluşturulması.*
* Kazandırılan: Vektör tabanlı arama, Retriever, Embedding kavramı.
* Kullanılan Bileşenler: FAISS, Chroma, RetrievalQA

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-text-splitters>=0.2" \
               transformers accelerate sentencepiece \
               pypdf faiss-cpu sentence-transformers

In [ ]:


# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
import os, textwrap, glob
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ============================================================
# 3) MODEL ve EMBEDDING
# ============================================================
# -- LLM (küçük ve hızlı): Qwen 0.5B Instruct
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # alternatif: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

# -- Embedding (hız/kalite dengesi iyi)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# ============================================================
# 4) DOKÜMANLARI YÜKLEME
#    /kaggle/input/altinda-belge-klasoru/ içindeki .txt, .md, .pdf dosyaları oku
# ============================================================
DOC_DIR = "/kaggle/input/rag-belgeleri"   # <-- kendi klasör yolunuzu girin

# DirectoryLoader PDF ve düz metinleri otomatik tespit etmez; pattern ile iki kez çağırıyoruz:
docs = []
if os.path.isdir(DOC_DIR):
    # TXT & MD
    if glob.glob(os.path.join(DOC_DIR, "**/*.txt"), recursive=True) or \
       glob.glob(os.path.join(DOC_DIR, "**/*.md"), recursive=True):
        loader_txt = DirectoryLoader(DOC_DIR, glob="**/*.txt", loader_cls=TextLoader, recursive=True, show_progress=True)
        loader_md  = DirectoryLoader(DOC_DIR, glob="**/*.md",  loader_cls=TextLoader, recursive=True, show_progress=True)
        docs += loader_txt.load() + loader_md.load()
    # PDF
    for pdf_path in glob.glob(os.path.join(DOC_DIR, "**/*.pdf"), recursive=True):
        try:
            docs += PyPDFLoader(pdf_path).load()
        except Exception as e:
            print(f"PDF okunamadı: {pdf_path} | Hata: {e}")
else:
    raise FileNotFoundError(f"Klasör bulunamadı: {DOC_DIR}")

print(f"✅ Yüklenen doküman sayısı: {len(docs)}")

# ============================================================
# 5) PARÇALAMA (Chunking)
# ============================================================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # içerik yapısına göre 500-1200 arası önerilir
    chunk_overlap=120,   # bağlam taşımak için küçük örtüşme
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(docs)
print(f"✅ Üretilen parça sayısı: {len(chunks)}")

# ============================================================
# 6) VEKTÖR VERİTABANI (FAISS) — OLUŞTUR & KAYDET
# ============================================================
INDEX_DIR = "/kaggle/working/faiss_index"

vectordb = FAISS.from_documents(chunks, embeddings)
os.makedirs(INDEX_DIR, exist_ok=True)
vectordb.save_local(INDEX_DIR)
print(f"✅ FAISS index kaydedildi: {INDEX_DIR}")

# (İsteyenler için: Sonradan yükleme)
# vectordb = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)

retriever = vectordb.as_retriever(search_kwargs={"k": 4})

# ============================================================
# 7) PROMPT ve RAG ZİNCİRİ
# ============================================================
prompt = PromptTemplate.from_template(textwrap.dedent("""
    Aşağıda bir kullanıcı sorusu ve belgeden getirilen ilgili parçalar (context) veriliyor.
    Sadece bu bağlamdan yararlanarak, akademik ve açık bir Türkçe ile yanıt ver.
    Bağlamda olmayan bilgi için "Metinde bu bilgi yer almıyor." de; uydurma yapma.

    Soru: {question}

    Bağlam:
    {context}

    Cevap:
"""))

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # kısa/orta context için yeterli
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

# ============================================================
# 8) SORU ÇALIŞTIR
# ============================================================
question = "Belgelerdeki ana problem tanımı nedir ve önerilen çözüm yaklaşımı nasıl özetlenmiştir?"
result = qa_chain({"query": question})

print("\n=== SORU ===\n", question)
print("\n=== YANIT ===\n", result["result"])

print("\n=== KAYNAK PARÇALAR ===")
for i, d in enumerate(result["source_documents"], 1):
    meta = d.metadata
    src = meta.get("source", "NA")
    page = meta.get("page", "NA")
    print(f"\n--- Parça {i} | Kaynak: {src} | Sayfa: {page} ---")
    print(textwrap.shorten(d.page_content.replace("\n", " "), width=300, placeholder="..."))


## 9. Çoklu Aracı Kullanan Ajan (Multi-Tool Agent)
* Amaç: Ajanların aynı anda birçok tool kullanarak karar vermesini göstermek.
* Senaryo: Kullanıcı bir şehir planı yapar, ajan hava durumu, harita, maliyet gibi kaynaklara başvurur.
* Kazandırılan: Çok modüllü karar süreçleri.
* Kullanılan Bileşenler: ZeroShotAgent, initialize_agent, Multiple Tools

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-experimental>=0.0.64" \
               transformers accelerate sentencepiece \
               duckduckgo-search wikipedia arxiv requests

# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

from langchain.agents import initialize_agent, AgentType

# Araçlar
from langchain_community.tools import DuckDuckGoSearchRun, RequestsGetTool
from langchain_experimental.tools.python.tool import PythonREPLTool

from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.tools.arxiv.tool import ArxivQueryRun

# İsteğe bağlı: daha deterministik davranış için uyarılar
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 3) LLM: Açık kaynak Qwen (küçük ve hızlı)
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # Alternatif: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,     # araç seçimi ve muhakeme için daha tutarlı
    top_p=0.9,
    repetition_penalty=1.05,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) ARAÇLARI OLUŞTURMA
# ============================================================
# 4.1 Web Araması
search_tool = DuckDuckGoSearchRun(name="WebSearch")

# 4.2 Wikipedia
wiki_api = WikipediaAPIWrapper(lang="tr", top_k_results=3, doc_content_chars_max=4000)
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api, name="WikipediaTR")

# 4.3 arXiv (bilimsel makale arama)
arxiv_api = ArxivAPIWrapper(top_k_results=3, doc_content_chars_max=4000)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_api, name="Arxiv")

# 4.4 HTTP GET (ham içerik çekme)
http_get_tool = RequestsGetTool(name="HttpGet")

# 4.5 Python REPL (hesaplama / küçük kodlar)
python_tool = PythonREPLTool(name="Python")

tools = [search_tool, wiki_tool, arxiv_tool, http_get_tool, python_tool]

# ============================================================
# 5) AGENT: ZERO_SHOT_REACT_DESCRIPTION
# ============================================================
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # ReAct tarzı akıl yürütme + araç çağrımı
    verbose=True,                 # eğitimsel: düşünme/ara adımlarını göster
    handle_parsing_errors=True,   # küçük modellerde sağlamlık
)

# ============================================================
# 6) ÇOKLU ARAÇ KULLANIMINI TETİKLEYEN KOMPOZİT SORU
# ============================================================
prompt_multi = """
CRISPR-Cas9 tekniğini 2-3 cümlede Türkçe açıkla (kısa ve teknik).
Ardından arXiv'de son 2 yılda CRISPR ile ilgili iki önemli makale başlığını ve yıllarını bul.
Son olarak Türkiye'de bu deneyin basit set-up maliyetini yaklaşık 25.000 TL varsayalım;
1 USD = 32.5 TL kabulüyle USD karşılığını Python ile hesaplayıp sonucu ver.
Gerek oldukça uygun araçları (Wikipedia, Arxiv, WebSearch, Python) kullan.
"""

print("\n================= SORU 1 (Multi-Tool) =================\n")
ans1 = agent.run(prompt_multi)
print("\n================= CEVAP 1 =================\n")
print(ans1)

# ============================================================
# 7) BAŞKA ÖRNEKLER (farklı araçları tetiklemek için)
# ============================================================

# 7.1 Sadece Wikipedia eğilimli bir soru
prompt_wiki = "Transformer mimarisini kısa anlat; 'Attention is All You Need' çalışmasının kısa adını ve yılını belirt."
print("\n================= SORU 2 (Wikipedia +/ veya WebSearch) =================\n")
ans2 = agent.run(prompt_wiki)
print("\n================= CEVAP 2 =================\n")
print(ans2)

# 7.2 Web araması ve HTTP GET: belirli bir sayfayı bulup çekme (örnek)
prompt_http = """
LangChain'in GitHub sayfasını bul ve HTTP GET ile sayfa başlığını çekip özetle.
Sadece başlığa ve kısa açıklamaya yer ver.
"""
print("\n================= SORU 3 (WebSearch + HttpGet) =================\n")
ans3 = agent.run(prompt_http)
print("\n================= CEVAP 3 =================\n")
print(ans3)

# 7.3 Python hesaplama: birim dönüşümü/hesap (ajanın Python aracı kullanması beklenir)
prompt_math = """
Bir aracın menzili 420 kilometre ise mil cinsinden yaklaşık değeri nedir?
1 mil = 1.60934 km kabul et ve Python ile hesapla.
"""
print("\n================= SORU 4 (PythonREPL) =================\n")
ans4 = agent.run(prompt_math)
print("\n================= CEVAP 4 =================\n")
print(ans4)
